In [14]:
# ============ CELL 1: SETUP AND GPU CHECK ============
print("="*60)
print("STEP 1: SETUP AND GPU CHECK")
print("="*60)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory: {total_memory:.2f} GB")
else:
    print("WARNING: CUDA not available. Training will be slow on CPU.")

# Clear cache
torch.cuda.empty_cache()
print("GPU cache cleared")
print()

STEP 1: SETUP AND GPU CHECK
PyTorch version: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
GPU Memory: 23.99 GB
GPU cache cleared



In [15]:
# ============ CELL 2: INSTALL PACKAGES ============
print("="*60)
print("STEP 2: INSTALL PACKAGES")
print("="*60)

# Install required packages
! pip install transformers datasets torch -q

print("Packages installed successfully!")
print()

STEP 2: INSTALL PACKAGES
Packages installed successfully!




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# ============ CELL 3: IMPORTS ============
print("="*60)
print("STEP 3: IMPORTS")
print("="*60)

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import re
import math
import os
import gc
from typing import List, Dict

print("All imports successful!")
print()

STEP 3: IMPORTS
All imports successful!



In [17]:
# ============ CELL 4: CONFIGURATION ============
print("="*60)
print("STEP 4: CONFIGURATION")
print("="*60)

config = {
    'model_name': 'gpt2',  # Start with small, stable model
    'max_length': 128,     # Short sequences for testing
    'max_new_tokens': 30,  # Short generation
    'epochs': 2,           # Few epochs for testing
    'batch_size': 1,       # Batch size 1 for memory safety
    'learning_rate': 1e-4, # Small learning rate
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key:20}: {value}")
print()

STEP 4: CONFIGURATION
Configuration:
  model_name          : gpt2
  max_length          : 128
  max_new_tokens      : 30
  epochs              : 2
  batch_size          : 1
  learning_rate       : 0.0001



In [18]:
# ============ CELL 5: SIMPLE REWARD CALCULATOR ============
print("="*60)
print("STEP 5: REWARD CALCULATOR")
print("="*60)

class SimpleRewardCalculator:
    """Simple reward calculator for testing"""
    def __init__(self):
        self.pattern = re.compile(r"<think>(.*?)</think>.*?<answer>(.*?)</answer>", re.DOTALL)
    
    def calculate_reward(self, response: str, ground_truth: str) -> Dict[str, float]:
        """Calculate reward for a response"""
        # Basic format check
        if "<think>" not in response or "<answer>" not in response:
            return {"total": 0.0, "format": 0.0, "accuracy": 0.0}
        
        # Format reward
        format_reward = 0.3
        
        # Try to extract and compare numbers
        try:
            # Find all numbers in response
            response_nums = re.findall(r"\d+", response)
            gt_nums = re.findall(r"\d+", ground_truth)
            
            if response_nums and gt_nums:
                # Check if any number matches
                matches = sum(1 for r in response_nums for g in gt_nums if r == g)
                if matches > 0:
                    accuracy = 0.7
                else:
                    accuracy = 0.2
            else:
                accuracy = 0.1
        except:
            accuracy = 0.1
        
        total = format_reward + accuracy
        
        return {
            "total": total,
            "format": format_reward,
            "accuracy": accuracy
        }

# Test the calculator
calculator = SimpleRewardCalculator()
test_response = "User: What is 2+2?\nAssistant: <think>Adding numbers</think><answer>4</answer>"
test_gt = "4"
reward = calculator.calculate_reward(test_response, test_gt)
print(f"Test reward calculation: {reward}")
print()

STEP 5: REWARD CALCULATOR
Test reward calculation: {'total': 1.0, 'format': 0.3, 'accuracy': 0.7}



In [19]:
# ============ CELL 6: CREATE SIMPLE DATASET ============
print("="*60)
print("STEP 6: CREATE DATASET")
print("="*60)

def create_simple_dataset(num_samples=50):
    """Create a simple math dataset for testing"""
    data = []
    
    for i in range(num_samples):
        # Create simple math problems
        a = random.randint(1, 20)
        b = random.randint(1, 20)
        operation = random.choice(['+', '-', '*'])
        
        if operation == '+':
            problem = f"What is {a} + {b}?"
            answer = a + b
        elif operation == '-':
            problem = f"What is {a} - {b}?"
            answer = a - b
        else:  # '*'
            problem = f"What is {a} × {b}?"
            answer = a * b
        
        # Format with think/answer tags
        text = f"User: {problem}\nAssistant: <think>Calculating {a} {operation} {b}</think><answer>{answer}</answer>"
        
        data.append({
            'problem': problem,
            'answer': str(answer),
            'text': text
        })
    
    return data

# Create dataset
print("Creating dataset...")
raw_data = create_simple_dataset(30)  # Small dataset for testing
print(f"Created {len(raw_data)} samples")

# Show examples
print("\nSample data:")
for i in range(min(2, len(raw_data))):
    print(f"  {raw_data[i]['text'][:80]}...")
print()

STEP 6: CREATE DATASET
Creating dataset...
Created 30 samples

Sample data:
  User: What is 16 - 6?
Assistant: <think>Calculating 16 - 6</think><answer>10</an...
  User: What is 10 × 20?
Assistant: <think>Calculating 10 * 20</think><answer>200<...



In [20]:
# ============ CELL 7: TOKENIZE DATASET ============
print("="*60)
print("STEP 7: TOKENIZE DATASET")
print("="*60)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {config['model_name']}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Tokenize dataset
tokenized_data = []
for item in raw_data:
    encoded = tokenizer(
        item['text'],
        truncation=True,
        max_length=config['max_length'],
        padding='max_length',
        return_tensors='pt'
    )
    
    tokenized_data.append({
        'input_ids': encoded['input_ids'][0],
        'attention_mask': encoded['attention_mask'][0],
        'text': item['text'],
        'ground_truth': item['answer']
    })

print(f"Tokenized {len(tokenized_data)} samples")
print(f"Input shape: {tokenized_data[0]['input_ids'].shape}")
print()

STEP 7: TOKENIZE DATASET


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded: gpt2
Vocabulary size: 50257
Tokenized 30 samples
Input shape: torch.Size([128])



In [21]:
# ============ CELL 8: CREATE DATASET CLASS ============
print("="*60)
print("STEP 8: CREATE DATASET CLASS")
print("="*60)

class MathDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': item['input_ids'].clone().detach(),
            'attention_mask': item['attention_mask'].clone().detach(),
            'text': item['text'],
            'ground_truth': item['ground_truth']
        }

# Create dataset
dataset = MathDataset(tokenized_data)
print(f"Dataset created with {len(dataset)} samples")
print()

STEP 8: CREATE DATASET CLASS
Dataset created with 30 samples



In [22]:
# ============ CELL 9: SIMPLE TRAINER CLASS ============
print("="*60)
print("STEP 9: CREATE TRAINER")
print("="*60)

class SimpleTrainer:
    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])
        self.reward_calculator = SimpleRewardCalculator()
    
    def train_step(self, batch):
        """Simple training step without generation"""
        # Forward pass
        outputs = self.model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['input_ids']  # Use input as labels for language modeling
        )
        
        loss = outputs.loss
        
        # Backward pass
        self.optimizer.zero_grad()
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        
        # Optimizer step
        self.optimizer.step()
        
        return loss.item()
    
    def generate_and_evaluate(self, batch):
        """Generate responses and calculate rewards"""
        with torch.no_grad():
            self.model.eval()
            
            # Generate response
            outputs = self.model.generate(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                max_new_tokens=config['max_new_tokens'],
                do_sample=True,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
            
            # Decode responses
            responses = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            # Calculate rewards
            rewards = []
            for response, gt in zip(responses, batch['ground_truth']):
                reward = self.reward_calculator.calculate_reward(response, gt)['total']
                rewards.append(reward)
            
            avg_reward = sum(rewards) / len(rewards) if rewards else 0.0
            
            return responses, avg_reward

print("Trainer class created")
print()

STEP 9: CREATE TRAINER
Trainer class created



In [23]:
# ============ CELL 10: MAIN TRAINING LOOP ============
print("="*60)
print("STEP 10: MAIN TRAINING")
print("="*60)

# Clear memory
torch.cuda.empty_cache()
gc.collect()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model
print(f"Loading model: {config['model_name']}...")
model = AutoModelForCausalLM.from_pretrained(config['model_name']).to(device)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Create trainer
trainer = SimpleTrainer(model, tokenizer, config)

# Create data loader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    collate_fn=lambda x: {
        'input_ids': torch.stack([item['input_ids'] for item in x]),
        'attention_mask': torch.stack([item['attention_mask'] for item in x]),
        'ground_truth': [item['ground_truth'] for item in x],
        'text': [item['text'] for item in x]
    }
)

print(f"\nStarting training...")
print(f"Epochs: {config['epochs']}")
print(f"Batch size: {config['batch_size']}")
print(f"Total batches: {len(dataloader)} per epoch")
print()

# Training loop
for epoch in range(config['epochs']):
    print(f"{'='*50}")
    print(f"EPOCH {epoch + 1}/{config['epochs']}")
    print(f"{'='*50}")
    
    epoch_loss = 0
    epoch_reward = 0
    batch_count = 0
    
    for batch_idx, batch in enumerate(dataloader):
        try:
            # Move batch to device
            batch = {
                'input_ids': batch['input_ids'].to(device),
                'attention_mask': batch['attention_mask'].to(device),
                'ground_truth': batch['ground_truth']
            }
            
            # Training step
            loss = trainer.train_step(batch)
            epoch_loss += loss
            batch_count += 1
            
            # Generate and evaluate every 5 batches
            if (batch_idx + 1) % 5 == 0:
                responses, avg_reward = trainer.generate_and_evaluate(batch)
                epoch_reward += avg_reward
                
                # Log progress
                current_loss = epoch_loss / 5
                print(f"  Batch {batch_idx + 1:3d} | "
                      f"Loss: {current_loss:.4f} | "
                      f"Reward: {avg_reward:.3f}")
                
                # Show sample response
                if responses:
                    print(f"    Sample: {responses[0][:80]}...")
                
                epoch_loss = 0
            
            # Clear cache periodically
            if (batch_idx + 1) % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            continue
    
    # End of epoch
    print(f"\nEpoch {epoch + 1} completed")
    
    # Generate test example
    print("\nTest generation:")
    try:
        test_input = "User: What is 10 + 5?"
        inputs = tokenizer(test_input, return_tensors="pt").to(device)
        
        with torch.no_grad():
            model.eval()
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=True,
                temperature=0.7
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"  Input: {test_input}")
        print(f"  Response: {response[:100]}...")
        
    except Exception as e:
        print(f"  Test generation failed: {e}")
    
    print()

print(f"{'='*50}")
print("TRAINING COMPLETE!")
print(f"{'='*50}")

STEP 10: MAIN TRAINING
Using device: cuda
Loading model: gpt2...


c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\user\.conda\envs\gpu_env\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid co

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Model loaded. Parameters: 124,439,808

Starting training...
Epochs: 2
Batch size: 1
Total batches: 30 per epoch

EPOCH 1/2
  Batch   5 | Loss: 3.7933 | Reward: 1.000
    Sample: User: What is 18 - 2?
Assistant: <think>Calculating 18 - 2</think><answer>16</an...
  Batch  10 | Loss: 0.3401 | Reward: 1.000
    Sample: User: What is 8 × 3?
Assistant: <think>Calculating 8 * 3</think><answer>24</answ...
  Batch  15 | Loss: 0.1342 | Reward: 1.000
    Sample: User: What is 16 - 6?
Assistant: <think>Calculating 16 - 6</think><answer>10</an...
  Batch  20 | Loss: 0.1476 | Reward: 1.000
    Sample: User: What is 19 × 19?
Assistant: <think>Calculating 19 * 19</think><answer>361<...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  Batch  25 | Loss: 0.1112 | Reward: 1.000
    Sample: User: What is 19 - 12?
Assistant: <think>Calculating 19 - 12</think><answer>7</a...
  Batch  30 | Loss: 0.1057 | Reward: 1.000
    Sample: User: What is 20 × 10?
Assistant: <think>Calculating 20 * 10</think><answer>200<...

Epoch 1 completed

Test generation:
  Input: User: What is 10 + 5?
  Response: User: What is 10 + 5?
Assistant: <think>Calculating 10 + 5</think><answer>5</answer>...

EPOCH 2/2
  Batch   5 | Loss: 0.0742 | Reward: 1.000
    Sample: User: What is 20 × 10?
Assistant: <think>Calculating 20 * 10</think><answer>200<...
  Batch  10 | Loss: 0.0970 | Reward: 1.000
    Sample: User: What is 2 × 14?
Assistant: <think>Calculating 2 * 14</think><answer>28</an...
  Batch  15 | Loss: 0.0892 | Reward: 1.000
    Sample: User: What is 15 - 10?
Assistant: <think>Calculating 15 - 10</think><answer>5</a...
  Batch  20 | Loss: 0.1044 | Reward: 1.000
    Sample: User: What is 5 - 13?
Assistant: <think>Calculating 5 - 13</think><answ

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  Batch  30 | Loss: 0.1007 | Reward: 1.000
    Sample: User: What is 19 × 19?
Assistant: <think>Calculating 19 * 19</think><answer>361<...

Epoch 2 completed

Test generation:
  Input: User: What is 10 + 5?
  Response: User: What is 10 + 5?
Assistant: <think>Calculating 10</think><answer>12</answer>...

TRAINING COMPLETE!


In [24]:
# ============ CELL 11: SAVE AND TEST MODEL ============
print("="*60)
print("STEP 11: SAVE AND TEST MODEL")
print("="*60)

# Save model
save_dir = "./trained_math_model"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved to: {save_dir}")

# Test final model
print("\nFinal model test:")
test_questions = [
    "What is 8 + 7?",
    "What is 15 - 6?",
    "What is 4 × 5?",
]

model.eval()
for question in test_questions:
    try:
        prompt = f"User: {question}\nAssistant:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Q: {question}")
        print(f"A: {response[len(prompt):]}")
        print()
        
    except Exception as e:
        print(f"Error generating for '{question}': {e}")

print("\n" + "="*60)
print("ALL DONE! Model trained and saved successfully.")
print("="*60)

STEP 11: SAVE AND TEST MODEL
Model saved to: ./trained_math_model

Final model test:
Q: What is 8 + 7?
A:  <think>Calculating 8 + 7</think><answer>26</answer>

Q: What is 15 - 6?
A:  <think>Calculating 15 - 6</think><answer>13</answer>

Q: What is 4 × 5?
A:  <think>Calculating 4 * 5</think><answer>9</answer>


ALL DONE! Model trained and saved successfully.
